# Daily Data Generation Notebook
## Purpose: Generate synthetic food delivery data for ONE day

**Note: This version uses Spark tables instead of DBFS (DBFS disabled in Free Edition)**

In [ ]:
# Databricks Daily Data Generation
# Uses Spark tables instead of DBFS (Free Edition compatible)

import uuid
import numpy as np
import pandas as pd
from datetime import date, datetime, timedelta
from faker import Faker
import json

# Configuration
CONFIG = {
    "seed": int(datetime.now().strftime("%Y%m%d")),
    "null_rate": 0.02,
    "duplicate_rate": 0.005,
    "sla_breach_rate": 0.12,
    "late_event_rate": 0.03,
    "orphan_rate": 0.01,
    "refund_rate": 0.08,
    "support_ticket_rate": 0.05
}

CITIES = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Hyderabad", "Pune", "Kolkata", "Ahmedabad"]

print(f"Starting daily data generation for {date.today()}")
np.random.seed(CONFIG["seed"])
Faker.seed(CONFIG["seed"])

In [ ]:
# Helper functions
def inject_nulls(df, columns, rate):
    df = df.copy()
    for col in columns:
        if col in df.columns:
            mask = np.random.random(len(df)) < rate
            df.loc[mask, col] = None
    return df

def inject_duplicates(df, rate):
    n_dups = int(len(df) * rate)
    if n_dups == 0:
        return df
    dup_indices = np.random.choice(df.index, n_dups, replace=True)
    duplicates = df.loc[dup_indices].copy()
    return pd.concat([df, duplicates], ignore_index=True)

In [ ]:
# Generate Restaurants (master data)
def generate_restaurants():
    faker = Faker("en_IN")
    cuisines = ["Biryani", "Chinese", "South Indian", "North Indian", "Fast Food", "Pizza", "Desserts", "Continental"]
    ratings = ["Excellent", "Good", "Average", "Poor"]
    
    restaurants = []
    for i in range(150):
        restaurants.append({
            "restaurant_id": f"REST-{i+1:05d}",
            "city": np.random.choice(CITIES),
            "cuisine_type": np.random.choice(cuisines),
            "rating_band": np.random.choice(ratings, p=[0.1, 0.4, 0.35, 0.15]),
            "onboarding_date": (date.today() - timedelta(days=np.random.randint(30, 365))).isoformat()
        })
    return pd.DataFrame(restaurants)

restaurants_df = generate_restaurants()
print(f"Generated {len(restaurants_df)} restaurants")
restaurants_df.head()

In [ ]:
# Generate Riders (master data)
def generate_riders():
    shifts = ["Morning", "Evening", "Night"]
    riders = []
    for i in range(300):
        riders.append({
            "rider_id": f"RIDER-{i+1:05d}",
            "city": np.random.choice(CITIES),
            "shift_type": np.random.choice(shifts),
            "joining_date": (date.today() - timedelta(days=np.random.randint(30, 730))).isoformat()
        })
    return pd.DataFrame(riders)

riders_df = generate_riders()
print(f"Generated {len(riders_df)} riders")

In [ ]:
# Generate Orders for ONE day
def generate_daily_orders(target_date, restaurants_df, num_orders_range=(800, 1500)):
    HOUR_WEIGHTS = [1,2,3,8,10,6, 3,3,4,7,10,8, 6,4,2,1]
    STATUS_WEIGHTS = {"delivered": 0.85, "cancelled": 0.10, "in_progress": 0.05}
    PAYMENT_MODES = {"UPI": 0.55, "Card": 0.20, "COD": 0.15, "Wallet": 0.10}
    
    base_volume = np.random.randint(*num_orders_range)
    dow_multiplier = 1.3 if target_date.weekday() >= 5 else 1.0
    daily_orders = int(base_volume * dow_multiplier)
    
    orders = []
    # Build restaurant lookup by city (compatible with older pandas)
    restaurant_by_city = {}
    for city in CITIES:
        city_df = restaurants_df[restaurants_df["city"] == city]
        restaurant_by_city[city] = city_df.to_dict("records")
    
    for _ in range(daily_orders):
        hours = list(range(8, 24))
        weights = np.array(HOUR_WEIGHTS) / sum(HOUR_WEIGHTS)
        hour = int(np.random.choice(hours, p=weights))
        minute = np.random.randint(0, 60)
        second = np.random.randint(0, 60)
        order_ts = datetime(target_date.year, target_date.month, target_date.day, hour, minute, second)
        
        city = np.random.choice(CITIES)
        city_restaurants = restaurant_by_city.get(city, [])
        restaurant = np.random.choice(city_restaurants) if city_restaurants else restaurants_df.sample(1).to_dict("records")[0]
        
        sla_minutes = np.random.randint(30, 45)
        promised_ts = order_ts + timedelta(minutes=sla_minutes)
        will_breach = np.random.random() < CONFIG["sla_breach_rate"]
        
        status = np.random.choice(list(STATUS_WEIGHTS.keys()), p=list(STATUS_WEIGHTS.values()))
        order_value = round(np.random.lognormal(5.5, 0.6), 2)
        order_value = max(50, min(order_value, 5000))
        payment_mode = np.random.choice(list(PAYMENT_MODES.keys()), p=list(PAYMENT_MODES.values()))
        
        orders.append({
            "order_id": f"ORD-{uuid.uuid4().hex[:12].upper()}",
            "customer_id": f"CUST-{np.random.randint(10000, 99999)}",
            "restaurant_id": restaurant["restaurant_id"],
            "city": city,
            "order_ts": order_ts.isoformat(),
            "promised_delivery_ts": promised_ts.isoformat(),
            "status": status,
            "order_value": order_value,
            "payment_mode": payment_mode,
            "_will_breach_sla": will_breach
        })
    
    df = pd.DataFrame(orders)
    df = inject_nulls(df, ["payment_mode"], CONFIG["null_rate"])
    df = inject_duplicates(df, CONFIG["duplicate_rate"])
    return df

target_date = date.today()
orders_df = generate_daily_orders(target_date, restaurants_df)
print(f"Generated {len(orders_df)} orders for {target_date}")

In [ ]:
# Generate Order Items
def generate_order_items(orders_df):
    cuisines = ["Biryani", "Chinese", "South Indian", "North Indian", "Fast Food", "Pizza", "Desserts", "Continental"]
    items = []
    for _, order in orders_df.iterrows():
        num_items = np.random.randint(1, 5)
        for j in range(num_items):
            items.append({
                "order_id": order["order_id"],
                "item_id": f"{order['order_id']}-ITEM-{j+1}",
                "quantity": np.random.randint(1, 4),
                "item_price": round(np.random.uniform(50, 500), 2),
                "cuisine_type": np.random.choice(cuisines)
            })
    return pd.DataFrame(items)

order_items_df = generate_order_items(orders_df)
print(f"Generated {len(order_items_df)} order items")

In [ ]:
# Generate Delivery Events
def generate_delivery_events(orders_df, riders_df):
    EVENT_SEQUENCE = ["order_confirmed", "restaurant_accepted", "food_prep_started", "food_ready", "rider_assigned", "rider_picked_up", "out_for_delivery", "delivered"]
    EVENT_DELTAS = {"order_confirmed": (0, 0), "restaurant_accepted": (1, 3), "food_prep_started": (2, 5), "food_ready": (10, 25), "rider_assigned": (1, 5), "rider_picked_up": (5, 15), "out_for_delivery": (0, 1), "delivered": (10, 20)}
    
    riders_by_city = riders_df.groupby("city").apply(lambda x: x["rider_id"].tolist()).to_dict()
    events = []
    
    for _, order in orders_df.iterrows():
        city = order.get("city", "Mumbai")
        status = order.get("status", "delivered")
        will_breach = order.get("_will_breach_sla", False)
        
        if status == "cancelled":
            n_events = np.random.randint(1, 4)
            event_chain = EVENT_SEQUENCE[:n_events] + ["cancelled"]
        else:
            event_chain = EVENT_SEQUENCE.copy()
        
        city_riders = riders_by_city.get(city, [])
        rider_id = np.random.choice(city_riders) if city_riders else ""
        current_ts = pd.to_datetime(order["order_ts"])
        
        for event_type in event_chain:
            min_d, max_d = EVENT_DELTAS.get(event_type, (1, 5))
            delta = np.random.randint(min_d, max_d + 1)
            if will_breach and event_type in ["food_ready", "delivered"]:
                delta += np.random.randint(10, 30)
            current_ts += timedelta(minutes=delta)
            
            event_rider = rider_id if event_type in ["rider_assigned", "rider_picked_up", "out_for_delivery", "delivered"] else ""
            events.append({
                "order_id": order["order_id"],
                "rider_id": event_rider,
                "event_type": event_type,
                "event_ts": current_ts.isoformat(),
                "latitude": round(19.076 + np.random.uniform(-0.1, 0.1), 6),
                "longitude": round(72.877 + np.random.uniform(-0.1, 0.1), 6)
            })
    
    # Inject orphans
    n_orphans = int(len(events) * CONFIG["orphan_rate"])
    orphan_indices = np.random.choice(len(events), n_orphans, replace=False)
    for i in orphan_indices:
        events[i]["order_id"] = f"ORD-INVALID-{uuid.uuid4().hex[:8].upper()}"
    
    return events

events = generate_delivery_events(orders_df, riders_df)
print(f"Generated {len(events)} delivery events")

In [ ]:
# Generate Refunds
def generate_refunds(orders_df):
    REFUND_REASONS = {"Delay": 0.35, "Missing_Items": 0.30, "Cancellation": 0.20, "Wrong_Order": 0.10, "Quality_Issue": 0.05}
    eligible_orders = orders_df[orders_df["status"] == "delivered"].copy()
    refund_orders = eligible_orders.sample(frac=CONFIG["refund_rate"])
    
    refunds = []
    for _, order in refund_orders.iterrows():
        reason = np.random.choice(list(REFUND_REASONS.keys()), p=list(REFUND_REASONS.values()))
        refund_ts = pd.to_datetime(order["order_ts"]) + timedelta(minutes=np.random.randint(30, 180))
        refunds.append({
            "refund_id": f"REF-{uuid.uuid4().hex[:12].upper()}",
            "order_id": order["order_id"],
            "refund_ts": refund_ts.isoformat(),
            "refund_reason": reason,
            "refund_amount": round(order["order_value"] * np.random.uniform(0.2, 1.0), 2)
        })
    return pd.DataFrame(refunds)

refunds_df = generate_refunds(orders_df)
print(f"Generated {len(refunds_df)} refunds")

In [ ]:
# Generate Support Tickets
def generate_support_tickets(orders_df):
    TICKET_TYPES = ["Complaint", "Inquiry", "Feedback", "Refund Request"]
    RESOLUTION_STATUS = ["resolved", "pending", "in_progress"]
    
    ticket_orders = orders_df.sample(frac=CONFIG["support_ticket_rate"])
    tickets = []
    for _, order in ticket_orders.iterrows():
        created_ts = pd.to_datetime(order["order_ts"]) + timedelta(minutes=np.random.randint(15, 120))
        tickets.append({
            "ticket_id": f"TKT-{uuid.uuid4().hex[:12].upper()}",
            "order_id": order["order_id"],
            "ticket_type": np.random.choice(TICKET_TYPES),
            "created_ts": created_ts.isoformat(),
            "resolution_status": np.random.choice(RESOLUTION_STATUS, p=[0.6, 0.3, 0.1])
        })
    return pd.DataFrame(tickets)

tickets_df = generate_support_tickets(orders_df)
print(f"Generated {len(tickets_df)} support tickets")

In [ ]:
# Save to Spark Tables (instead of DBFS)
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit

spark = SparkSession.builder.getOrCreate()
batch_id = datetime.now().strftime("%Y%m%d_%H%M%S")

print("\n--- Saving to Spark Tables ---")

# Save restaurants (master - overwrite)
restaurants_sdf = spark.createDataFrame(restaurants_df)\
    .withColumn("_batch_id", lit(batch_id))\
    .withColumn("_ingested_at", current_timestamp())
restaurants_sdf.write.mode("overwrite").saveAsTable("bronze_restaurants")
print(f"✓ bronze_restaurants: {restaurants_sdf.count()} rows")

# Save riders (master - overwrite)
riders_sdf = spark.createDataFrame(riders_df)\
    .withColumn("_batch_id", lit(batch_id))\
    .withColumn("_ingested_at", current_timestamp())
riders_sdf.write.mode("overwrite").saveAsTable("bronze_riders")
print(f"✓ bronze_riders: {riders_sdf.count()} rows")

# Save orders (daily - append)
orders_sdf = spark.createDataFrame(orders_df.drop(columns=["_will_breach_sla"]))\
    .withColumn("_batch_id", lit(batch_id))\
    .withColumn("_ingested_at", current_timestamp())
orders_sdf.write.mode("append").saveAsTable("bronze_orders")
print(f"✓ bronze_orders: {orders_sdf.count()} rows")

# Save order_items (daily - append)
order_items_sdf = spark.createDataFrame(order_items_df)\
    .withColumn("_batch_id", lit(batch_id))\
    .withColumn("_ingested_at", current_timestamp())
order_items_sdf.write.mode("append").saveAsTable("bronze_order_items")
print(f"✓ bronze_order_items: {order_items_sdf.count()} rows")

# Save delivery_events (daily - append)
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

events_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("rider_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_ts", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True)
])

events_sdf = spark.createDataFrame(events, schema=events_schema)\
    .withColumn("_batch_id", lit(batch_id))\
    .withColumn("_ingested_at", current_timestamp())
events_sdf.write.mode("append").saveAsTable("bronze_delivery_events")
print(f"✓ bronze_delivery_events: {events_sdf.count()} rows")

# Save refunds (daily - append)
refunds_sdf = spark.createDataFrame(refunds_df)\
    .withColumn("_batch_id", lit(batch_id))\
    .withColumn("_ingested_at", current_timestamp())
refunds_sdf.write.mode("append").saveAsTable("bronze_refunds")
print(f"✓ bronze_refunds: {refunds_sdf.count()} rows")

# Save support_tickets (daily - append)
tickets_sdf = spark.createDataFrame(tickets_df)\
    .withColumn("_batch_id", lit(batch_id))\
    .withColumn("_ingested_at", current_timestamp())
tickets_sdf.write.mode("append").saveAsTable("bronze_support_tickets")
print(f"✓ bronze_support_tickets: {tickets_sdf.count()} rows")

In [ ]:
# Summary
print("\n" + "="*60)
print(f"DAILY DATA GENERATION COMPLETE for {target_date}")
print("="*60)
print(f"\nSpark Tables created:")
print(f"  - bronze_restaurants (master)")
print(f"  - bronze_riders (master)")
print(f"  - bronze_orders (daily)")
print(f"  - bronze_order_items (daily)")
print(f"  - bronze_delivery_events (daily)")
print(f"  - bronze_refunds (daily)")
print(f"  - bronze_support_tickets (daily)")
print(f"\nBatch ID: {batch_id}")